# Task B Run 26 -- Gemma-4-12B LoRA capacity ablation

Run 21 showed that Gemma-4-12B QLoRA is substantially stronger than the MuRIL recipe on the fixed combined holdout. This run asks a narrower, Gemma-only question: how much adapter capacity is useful for this small corpus?

| arm | LoRA rank | adapted modules |
|---|---:|---|
| attention-only | 8, 16, 32 | `q_proj,k_proj,v_proj,o_proj` |
| attention + MLP | 8, 16, 32 | attention plus `gate_proj,up_proj,down_proj` |

Every arm is trained with seeds **42 and 43** on the same committed 15% holdout split, with the same Gemma checkpoint, prompt, optimizer, three epochs, sequence length and class weighting. The two seed probability matrices are averaged within each arm. We report macro-F1, accuracy, per-class F1, trainable-parameter diagnostics from the logs, and paired bootstrap intervals against the current `r=16` attention+MLP control.

This is an evaluation-only notebook. It deliberately does **not** train on all rows or create a test ZIP: selecting an arm on this holdout makes its holdout score optimistic. After the capacity choice is recorded, a separate full-data Gemma notebook should train the selected recipe and generate the submission.

## Before running

- Upload this notebook to Kaggle with **GPU T4 x2**, Internet enabled and the Hugging Face `HF_TOKEN` secret available.
- Save and run all. The twelve Gemma fits are queued two at a time and should take roughly 8--12 hours, depending on cache and GPU availability.
- Download `b24_capacity_outputs/` after the run; it contains one log, metrics file and holdout probability matrix per completed seed.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, time

REPO = "https://github.com/robinpnalex/Hastika-ICON2026.git"
BRANCH = "task-b"
WORK = "/kaggle/working/hastika"

def git(*args):
    return subprocess.run(["git", "-C", WORK, *args], check=True,
                          capture_output=True, text=True).stdout.strip()

try:
    remote = subprocess.run(["git", "ls-remote", "--exit-code", REPO,
                             f"refs/heads/{BRANCH}"], check=True,
                            capture_output=True, text=True, timeout=60).stdout.split()[0]
except Exception as e:
    raise SystemExit("Cannot reach GitHub. Turn on Kaggle Internet and restart the session.") from e

if os.path.isdir(WORK + "/.git"):
    git("fetch", "-q", "--depth", "1", "origin", BRANCH)
    git("reset", "-q", "--hard", "FETCH_HEAD")
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "--depth", "1", REPO, WORK], check=True)
head = git("rev-parse", "HEAD")
assert head == remote, f"clone is at {head[:8]} but {BRANCH} is at {remote[:8]}"
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret loaded")
except Exception:
    print("HF_TOKEN was not found; Gemma download may fail if the license is not public for this account")

# Install repo-level preprocessing dependencies before importing hastika.*.
# Kaggle's base image does not include ftfy, which preprocessing.py imports.
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)

import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), "select a GPU accelerator"
N_GPU = torch.cuda.device_count()
print("repo", git("log", "-1", "--oneline"))
print("gpus:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
if N_GPU < 2:
    print("WARNING: one GPU detected; the 12 jobs will run sequentially")
print(f"free /tmp: {shutil.disk_usage('/tmp').free / 2**30:.0f} GB")

OUT = pathlib.Path("/kaggle/working/b24_capacity_outputs")
RUNS, LOGS = OUT / "runs", OUT / "logs"
for p in (RUNS, LOGS):
    p.mkdir(parents=True, exist_ok=True)
HF_CACHE = "/tmp/hf"
os.environ["HF_HUB_CACHE"] = HF_CACHE

from hastika.common.gpu_queue import run_queue
from hastika.task_b.combined_holdout import LABELS, fingerprint, paired_bootstrap, report

SPLIT = pathlib.Path(WORK) / "data/derived/task_b_combined_holdout"
all_df = pd.read_csv(SPLIT / "all.csv")
train_df = pd.read_csv(SPLIT / "train.csv")
holdout = pd.read_csv(SPLIT / "holdout.csv")
assert (len(all_df), len(train_df), len(holdout)) == (3532, 3002, 530), (len(all_df), len(train_df), len(holdout))
HOLDOUT_FP = fingerprint(holdout["id"])
assert HOLDOUT_FP == "f85f4f049b", HOLDOUT_FP
y_ho = holdout["Hate Category"].map(LABELS.index).to_numpy()
print(f"train {len(train_df)}, holdout {len(holdout)}, fingerprint {HOLDOUT_FP}")

In [ ]:
# Gemma-4 requires the newer LLM stack, isolated from the repo's MuRIL environment.
LLM_PY = "/tmp/llmenv/bin/python"
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip", "/tmp/llmenv"], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q "transformers>=5.12,<6" "peft>=0.17" "bitsandbytes>=0.46" "accelerate>=1.0" emoji ftfy sentencepiece protobuf', shell=True, check=True)
subprocess.run([LLM_PY, "-c", "import torch, transformers, peft, bitsandbytes; print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__)"], check=True)

MODEL = "google/gemma-4-12B"
RANKS = (8, 16, 32)
SEEDS = (42, 43)
EPOCHS = 3
ATTN = "q_proj,k_proj,v_proj,o_proj"
FULL = "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj"
TARGETS = {"attn": ATTN, "full": FULL}

def download(model):
    from huggingface_hub import snapshot_download
    snapshot_download(model, cache_dir=HF_CACHE, token=os.environ.get("HF_TOKEN") or None,
                      allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*", "*.jinja"])
    print("downloaded", model, "| free /tmp", f"{shutil.disk_usage('/tmp').free / 2**30:.0f} GB")

download(MODEL)

def job_name(rank, target, seed):
    return f"gemma_r{rank}_{target}_s{seed}"

def llm_job(rank, target, seed):
    name = job_name(rank, target, seed)
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_classifier --model {MODEL} "
                    f"--train {SPLIT / 'train.csv'} --eval {SPLIT / 'holdout.csv'} "
                    f"--predict {SPLIT / 'holdout.csv'} --out {RUNS / name} "
                    f"--seed {seed} --epochs {EPOCHS} --r {rank} "
                    f"--lora-targets {TARGETS[target]} --cache {HF_CACHE}")}

jobs = [llm_job(r, t, s) for t in ("attn", "full") for r in RANKS for s in SEEDS]
print("queued arms:", [j["name"] for j in jobs])

## Train the twelve fixed-holdout fits

The queue pins one process to each visible GPU. It does not select checkpoints from the holdout: the runner trains the prescribed three epochs and the final model writes probabilities. If a T4 runs out of memory, the retry halves the micro-batch while preserving the effective batch size.

In [ ]:
def run_with_oom_retry(jobs, stop_at):
    codes = run_queue(jobs, N_GPU, raise_on_fail=False, stop_at=stop_at)
    retry = []
    for j in jobs:
        if codes.get(j["name"]) and "OutOfMemoryError" in open(j["log"], errors="replace").read():
            print("retrying", j["name"], "with smaller micro-batch")
            retry.append({**j, "cmd": j["cmd"] + " --bs 4 --grad-accum 4 --eval-bs 8"})
    if retry:
        codes.update(run_queue(retry, N_GPU, raise_on_fail=False, stop_at=stop_at))
    return codes

T_START = time.time()
HARD_STOP = T_START + 11 * 3600
codes = run_with_oom_retry(jobs, HARD_STOP)
print("exit codes:", codes)
print(f"elapsed {(time.time() - T_START) / 3600:.1f} h")

In [ ]:
def load_probs(name):
    p = RUNS / name / "holdout_probs.npy"
    return np.load(p) if p.exists() else None

results = {}
for target in ("attn", "full"):
    for rank in RANKS:
        members = {s: load_probs(job_name(rank, target, s)) for s in SEEDS}
        finished = {s: p for s, p in members.items() if p is not None}
        if not finished:
            print("missing", target, rank)
            continue
        assert all(p.shape == (len(holdout), 6) for p in finished.values())
        results[f"r{rank}_{target}"] = {"members": finished, "mean": np.mean(list(finished.values()), 0)}
        for seed, p in finished.items():
            report(y_ho, p, f"r{rank} {target} seed {seed}")
        report(y_ho, results[f"r{rank}_{target}"]["mean"], f"r{rank} {target} mean")

assert results, "no completed probability files; inspect b24_capacity_outputs/logs"
scores = {k: report(y_ho, v["mean"], k) for k, v in results.items()}
print("scores:", json.dumps(scores, indent=2))

## Paired uncertainty and selection

The comparison is paired because every arm predicts exactly the same holdout rows. The bootstrap is useful for judging whether a small capacity difference is larger than row-sampling noise; it is not a substitute for a fresh confirmation split. The `r16_full` arm is the pre-existing Gemma recipe and is the control for this ablation.

In [ ]:
CONTROL = "r16_full"
bootstraps = {}
if CONTROL in results:
    for name, item in results.items():
        if name == CONTROL:
            continue
        b = paired_bootstrap(y_ho, item["mean"], results[CONTROL]["mean"], n=5000, seed=2400)
        bootstraps[name] = b
        print(f"{name} - {CONTROL}: {b['diff']:+.4f}; CI [{b['ci95'][0]:+.4f}, {b['ci95'][1]:+.4f}]; P(better) {b['p_better']:.2f}")
else:
    print("control did not finish; bootstrap comparisons skipped")

CHOICE = max(scores, key=scores.get)
print(f"selected holdout arm: {CHOICE} ({scores[CHOICE]:.4f})")
print("Reminder: this choice is holdout-selected and therefore optimistic. Confirm it before paper claims.")

In [ ]:
# Keep a compact machine-readable record beside the downloaded logs and probabilities.
record = {
    "commit": head,
    "model": MODEL,
    "holdout_rows": len(holdout),
    "holdout_fingerprint": HOLDOUT_FP,
    "ranks": list(RANKS),
    "targets": TARGETS,
    "seeds": list(SEEDS),
    "epochs": EPOCHS,
    "scores": scores,
    "bootstrap_vs_r16_full": bootstraps,
    "choice": CHOICE,
    "exit_codes": codes,
    "completed_models": {k: sorted(map(int, v["members"].keys())) for k, v in results.items()},
}
json.dump(record, open(OUT / "result.json", "w"), indent=2)
print(json.dumps(record, indent=2))
print("No submission ZIP is created by this notebook; use the selected arm in a separate full-data run.")